# Results Analysis

In [ ]:
# Step 1: Setup
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("/Users/stevencamacho/Desktop/NLP_project/results")

DATASETS = [
    "natural_disasters",
    "conflicts",
    "infrastructure_problems",
]

OUTPUT_PATH = RESULTS_DIR / "asymmetry_index.csv"

In [ ]:
# Step 2: Load entity counts
frames = []
for dataset in DATASETS:

    candidates = [
        RESULTS_DIR / dataset / "analysis" / "entity_counts.csv",
        RESULTS_DIR / dataset / "analysis" / "entity_counts (1).csv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        print(f"  WARNING: no entity_counts.csv found for '{dataset}', skipping.")
        continue
    df = pd.read_csv(path)
    df["dataset"] = dataset
    frames.append(df)
    print(f"  Loaded {len(df)} rows from {dataset}")

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows loaded: {len(raw)}")
print(raw.head(6).to_string())

In [ ]:
# Step 3 Compute asymmetry index

raw[["model", "lang"]] = raw["model_lang"].str.rsplit("_", n=1, expand=True)

counts = raw[["dataset", "event", "model", "lang", "model_count"]]

# Pivot so EN and ES counts are in the same row
pivot = counts.pivot_table(
    index=["dataset", "event", "model"],
    columns="lang",
    values="model_count"
).reset_index()
pivot.columns.name = None

pivot = pivot.rename(columns={"en": "count_en", "es": "count_es"})

# Asymmetry index: (EN - ES) / (EN + ES)
# Returns None where denominator is 0 (both counts are 0)
def asymmetry_index(row):
    total = row["count_en"] + row["count_es"]
    if total == 0:
        return None
    return round((row["count_en"] - row["count_es"]) / total, 4)

pivot["asymmetry_index"] = pivot.apply(asymmetry_index, axis=1)

print(pivot.to_string())

In [ ]:
# Step 4: General Statistics
summary = (
    pivot
    .groupby(["dataset", "model"])["asymmetry_index"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        max="max",
        count="count"
    )
    .round(4)
    .reset_index()
)

print(summary.to_string())

                   dataset  model    mean     std     min     max  count
0                conflicts  gemma -0.0743  0.1401 -0.2500  0.2632     10
1                conflicts  llama  0.2496  0.1862 -0.0476  0.5676     10
2                conflicts   qwen  0.1454  0.2768 -0.1020  0.8462     10
3  infrastructure_problems  gemma  0.0633  0.1428 -0.1304  0.3333     10
4  infrastructure_problems  llama  0.0383  0.1968 -0.2308  0.3333     10
5  infrastructure_problems   qwen  0.1704  0.2357 -0.2000  0.5556     10
6        natural_disasters  gemma  0.0055  0.1547 -0.3333  0.2800     10
7        natural_disasters  llama  0.0896  0.1875 -0.0968  0.4667     10
8        natural_disasters   qwen  0.0929  0.2145 -0.2941  0.3750     10


In [ ]:
# Step 5: Mean and Std of Asymmetry Index per (Model, Genre)

# A. Mean asymmetry per (model, genre)
mean_asymmetry = (
    pivot
    .groupby(["model", "dataset"])["asymmetry_index"]
    .mean()
    .round(4)
    .reset_index()
    .rename(columns={"asymmetry_index": "mean_asymmetry", "dataset": "genre"})
)

# B. Standard deviation per (model, genre)
std_asymmetry = (
    pivot
    .groupby(["model", "dataset"])["asymmetry_index"]
    .std()
    .round(4)
    .reset_index()
    .rename(columns={"asymmetry_index": "std_asymmetry", "dataset": "genre"})
)

# Combine A and B into one table
model_genre_stats = mean_asymmetry.merge(std_asymmetry, on=["model", "genre"])

print("Mean and Std of Asymmetry Index per (Model, Genre)")
print("=" * 60)
print(model_genre_stats.to_string(index=False))

# Save
stats_path = RESULTS_DIR / "asymmetry_by_model_genre.csv"
model_genre_stats.to_csv(stats_path, index=False)
print(f"\nSaved -> {stats_path}")

Mean and Std of Asymmetry Index per (Model, Genre)
model                   genre  mean_asymmetry  std_asymmetry
gemma               conflicts         -0.0743         0.1401
gemma infrastructure_problems          0.0633         0.1428
gemma       natural_disasters          0.0055         0.1547
llama               conflicts          0.2496         0.1862
llama infrastructure_problems          0.0383         0.1968
llama       natural_disasters          0.0896         0.1875
 qwen               conflicts          0.1454         0.2768
 qwen infrastructure_problems          0.1704         0.2357
 qwen       natural_disasters          0.0929         0.2145

Saved -> /Users/stevencamacho/Desktop/NLP_project/results/asymmetry_by_model_genre.csv


In [ ]:
# Step 6: Bar chart of mean asymmetry index per genre and model
import matplotlib.pyplot as plt
import numpy as np

genres = model_genre_stats["genre"].unique()
models = model_genre_stats["model"].unique()
genre_labels = [g.replace("_", " ").title() for g in genres]

n_genres = len(genres)
n_models = len(models)
bar_width = 0.2
x = np.arange(n_genres)

fig, ax = plt.subplots(figsize=(11, 7))

for i, model in enumerate(models):
    model_data = model_genre_stats[model_genre_stats["model"] == model]
    means = []
    for genre in genres:
        row = model_data[model_data["genre"] == genre]
        means.append(row["mean_asymmetry"].values[0] if len(row) else np.nan)

    offset = (i - n_models / 2 + 0.5) * bar_width
    ax.bar(x + offset, means, width=bar_width, label=model.capitalize())

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(genre_labels, fontsize=16)
ax.tick_params(axis="y", labelsize=14)
ax.set_xlabel("Genre", fontsize=17)
ax.set_ylabel("Mean Asymmetry Index", fontsize=17)
ax.set_title(
    "Mean Asymmetry Index per Genre and Model\n"
    r" $= (N_{EN} - N_{ES})\ /\ (N_{EN} + N_{ES})$",
    fontsize=18
)
ax.legend(title="Model", fontsize=14, title_fontsize=15)
plt.tight_layout()

bar_path = RESULTS_DIR / "asymmetry_bar_chart.png"
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
print(f"Saved -> {bar_path}")
plt.show()


SyntaxError: unterminated string literal (detected at line 33) (3756614176.py, line 33)

In [ ]:
# Step 7: Boxplot of asymmetry index distribution per genre and model

genres  = pivot["dataset"].unique()
models  = pivot["model"].unique()
genre_labels = [g.replace("_", " ").title() for g in genres]

n_genres = len(genres)
n_models = len(models)

fig, axes = plt.subplots(1, n_genres, figsize=(6 * n_genres, 7), sharey=True)

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
model_colors = {model: prop_cycle[i] for i, model in enumerate(models)}

for ax, genre, label in zip(axes, genres, genre_labels):
    genre_data = pivot[pivot["dataset"] == genre]

    box_data   = []
    tick_labels = []
    box_colors = []

    for model in models:
        values = genre_data[genre_data["model"] == model]["asymmetry_index"].dropna().tolist()
        if values:
            box_data.append(values)
            tick_labels.append(model.capitalize())
            box_colors.append(model_colors[model])

    bp = ax.boxplot(
        box_data,
        tick_labels=tick_labels,
        patch_artist=True,
        medianprops={"color": "black", "linewidth": 2},
    )
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title(label, fontsize=17)
    ax.set_xlabel("Model", fontsize=15)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    if ax == axes[0]:
        ax.set_ylabel("Asymmetry Index", fontsize=16)

# Legend — one coloured patch per model
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=model_colors[m], alpha=0.7, label=m.capitalize())
    for m in models
]
fig.legend(
    handles=legend_handles,
    title="Model",
    loc="lower center",
    ncol=n_models,
    fontsize=14,
    title_fontsize=15,
    bbox_to_anchor=(0.5, -0.08),
)

fig.suptitle(
    "Distribution of Asymmetry Index per Genre and Model
"
    r" = (N_{EN} - N_{ES})\ /\ (N_{EN} + N_{ES})$",
    fontsize=18, y=1.02
)
plt.tight_layout()

box_path = RESULTS_DIR / "asymmetry_boxplot.png"
plt.savefig(box_path, dpi=150, bbox_inches="tight")
print(f"Saved -> {box_path}")
plt.show()


In [ ]:
# Step 8: Stacked bar chart of entity type proportions per model and language, grouped by genre
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATASETS      = ["conflicts", "natural_disasters"]
ENTITY_TYPES  = ["person", "organization", "location", "country", "date"]
TYPE_COLORS   = {
    "person":       "#4C72B0",
    "organization": "#DD8452",
    "location":     "#55A868",
    "country":      "#C44E52",
    "date":         "#8172B2",
}

fig, axes = plt.subplots(1, len(DATASETS), figsize=(8 * len(DATASETS), 8), sharey=True)

for ax, dataset in zip(axes, DATASETS):
    path = RESULTS_DIR / dataset / "analysis" / "entity_type_counts.csv"
    df = pd.read_csv(path)

    # Split model_lang into model and lang columns
    df[["model", "lang"]] = df["model_lang"].str.rsplit("_", n=1, expand=True)

    # Sum counts across all events per (model, lang)
    agg = df.groupby(["model", "lang"])[ENTITY_TYPES].sum().reset_index()

    # Compute row-wise proportions
    agg["total"] = agg[ENTITY_TYPES].sum(axis=1)
    for t in ENTITY_TYPES:
        agg[f"{t}_pct"] = agg[t] / agg["total"]

    # Build x-axis labels: one bar per model×lang, grouped by model
    models = sorted(agg["model"].unique())
    langs  = ["en", "es"]
    bar_labels = []
    bar_data   = {t: [] for t in ENTITY_TYPES}

    for model in models:
        for lang in langs:
            row = agg[(agg["model"] == model) & (agg["lang"] == lang)]
            bar_labels.append(f"{model.capitalize()}
{lang.upper()}")
            for t in ENTITY_TYPES:
                bar_data[t].append(row[f"{t}_pct"].values[0] if len(row) else 0)

    x = np.arange(len(bar_labels))
    bar_width = 0.55
    bottom = np.zeros(len(bar_labels))

    for t in ENTITY_TYPES:
        values = np.array(bar_data[t])
        ax.bar(x, values, bar_width, bottom=bottom,
               label=t.capitalize(), color=TYPE_COLORS[t])
        bottom += values

    # Dividers between model groups
    for i in range(1, len(models)):
        ax.axvline(i * len(langs) - 0.5, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(bar_labels, fontsize=13)
    ax.tick_params(axis="y", labelsize=13)
    ax.set_title(dataset.replace("_", " ").title(), fontsize=17, pad=12)
    ax.set_xlabel("Model / Language", fontsize=15)
    if ax == axes[0]:
        ax.set_ylabel("Proportion of Entity Types", fontsize=15)
    ax.set_ylim(0, 1)

# Single shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Entity Type", loc="lower center",
           ncol=len(ENTITY_TYPES), fontsize=13, title_fontsize=14,
           bbox_to_anchor=(0.5, -0.06))

fig.suptitle(
    "Entity Type Proportions by Model and Language per Genre",
    fontsize=19, y=1.01
)
plt.tight_layout(pad=1.5)
plt.savefig(RESULTS_DIR / "entity_type_proportions.png", dpi=150,
            bbox_inches="tight", pad_inches=0.3)
print("Saved -> entity_type_proportions.png")
plt.show()


In [ ]:
# Step 9: Compute precision of entity type counts against ground truth
import pandas as pd
from pathlib import Path

DATASETS     = ["conflicts", "natural_disasters", "infrastructure_problems"]
ENTITY_TYPES = ["person", "organization", "location", "country", "date"]

def load_csv(path):
    for suffix in ["", " (1)"]:
        p = Path(str(path).replace(".csv", f"{suffix}.csv")) if suffix else path
        if p.exists():
            return pd.read_csv(p)
    return None

rows = []

for dataset in DATASETS:
    model_path = RESULTS_DIR / dataset / "analysis" / "entity_type_counts.csv"
    gt_path    = RESULTS_DIR / dataset / "ground_truth" / "gt_entity_counts.csv"

    df_model = load_csv(model_path)
    df_gt    = load_csv(gt_path)

    if df_model is None or df_gt is None:
        print(f"WARNING: missing files for {dataset}")
        continue

    for model_lang, group in df_model.groupby("model_lang"):
        model_rows = group.reset_index(drop=True)
        gt_rows    = df_gt.reset_index(drop=True)

        # Align positionally — both have one row per event in the same order
        n_events = min(len(model_rows), len(gt_rows))

        precisions = []
        for i in range(n_events):
            m_counts = model_rows.loc[i, ENTITY_TYPES]
            g_counts = gt_rows.loc[i, ENTITY_TYPES]

            m_total = m_counts.sum()
            if m_total == 0:
                continue  # no model entities — skip event

            hits = sum(min(m_counts[t], g_counts[t]) for t in ENTITY_TYPES)
            precisions.append(hits / m_total)

        model, lang = model_lang.rsplit("_", 1)
        rows.append({
            "dataset":        dataset,
            "model":          model,
            "lang":           lang,
            "precision_mean": round(pd.Series(precisions).mean(), 4),
            "precision_std":  round(pd.Series(precisions).std(), 4),
            "n_events":       len(precisions),
        })

df_precision = pd.DataFrame(rows).sort_values(["dataset", "model", "lang"])
print(df_precision.to_string(index=False))

prec_path = RESULTS_DIR / "type_precision_by_model_lang_genre.csv"
df_precision.to_csv(prec_path, index=False)
print(f"Saved -> {prec_path}")


                dataset model lang  precision_mean  precision_std  n_events
              conflicts gemma   en          0.7417         0.3253        10
              conflicts gemma   es          0.7105         0.3481        10
              conflicts llama   en          0.5758         0.3013        10
              conflicts llama   es          0.7418         0.3518        10
              conflicts  qwen   en          0.7035         0.3577        10
              conflicts  qwen   es          0.7145         0.3837         9
infrastructure_problems gemma   en          0.5074         0.2046        10
infrastructure_problems gemma   es          0.4433         0.2434        10
infrastructure_problems llama   en          0.3953         0.1585        10
infrastructure_problems llama   es          0.3903         0.1593        10
infrastructure_problems  qwen   en          0.4734         0.2539        10
infrastructure_problems  qwen   es          0.6400         0.2493        10
      natura

In [ ]:
# Step 10: Save asymmetry index results
pivot.to_csv(OUTPUT_PATH, index=False)
print(f"Saved asymmetry index -> {OUTPUT_PATH}")
print(pivot[["dataset", "event", "model", "count_en", "count_es", "asymmetry_index"]].to_string())

Saved asymmetry index -> /Users/stevencamacho/Desktop/NLP_project/results/asymmetry_index.csv
                    dataset                                     event  model  count_en  count_es  asymmetry_index
0                 conflicts        2005 Bangladesh–India border clash  gemma      12.0       7.0           0.2632
1                 conflicts        2005 Bangladesh–India border clash  llama      10.0      11.0          -0.0476
2                 conflicts        2005 Bangladesh–India border clash   qwen      10.0      10.0           0.0000
3                 conflicts                          2006 Lebanon War  gemma      12.0      14.0          -0.0769
4                 conflicts                          2006 Lebanon War  llama      29.0       8.0           0.5676
5                 conflicts                          2006 Lebanon War   qwen      13.0       8.0           0.2381
6                 conflicts                  Central African Bush War  gemma      27.0      32.0          -0